In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(4048, 31)


In [2]:
interaction = (
    df.groupby(
        ["customerId", "skuNumber"]
    )["effective_qty"]
    .sum()
    .reset_index()
)

interaction.head()

,customerId,skuNumber,effective_qty
0,USR-1002,SKU00001,2
1,USR-1002,SKU00033,1
2,USR-1002,SKU080,1
3,USR-100211,SKU479,1
4,USR-100211,SKU482,1


In [3]:
print(
    "Interactions:",
    len(interaction)
)

print(
    "Retailers:",
    interaction["customerId"].nunique()
)

print(
    "Products:",
    interaction["skuNumber"].nunique()
)

Interactions: 3947
Retailers: 1350
Products: 159


In [4]:
matrix = interaction.pivot(
    index="customerId",
    columns="skuNumber",
    values="effective_qty"
).fillna(0)

matrix.shape

(1350, 159)

In [5]:
zeros = (matrix == 0).sum().sum()

total = (
    matrix.shape[0]
    * matrix.shape[1]
)

sparsity = (
    zeros / total
) * 100

print(
    f"Sparsity: {sparsity:.2f}%"
)

Sparsity: 98.16%


In [6]:
matrix.to_parquet(
    "../data/processed/interaction_matrix.parquet"
)

print("Saved")

Saved


In [7]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName",
            "brand",
            "category"
        ]
    ]
    .drop_duplicates()
)

sku_lookup.to_parquet(
    "../data/processed/sku_lookup.parquet",
    index=False
)

print("Saved")

Saved


In [8]:
top_products = (
    df.groupby("itemName")
    ["effective_qty"]
    .sum()
    .sort_values(
        ascending=False
    )
    .head(20)
)

top_products

itemName
Rajnigandha 17g Zipper 1 Pcs             1327
TZ 00 (with Silver) 0.45g                 615
Rajnigandha 4g                            486
TZ 00 4.25g (6 Pcs)                       277
RG Pearl Elaichi MP Pouch 0.14g           260
Rajnigandha 100g                          233
RG Pearl Elaichi 1.4g Hanger              188
Pass Pass Meetha Magic 2.2g (100 pcs)     169
Pass Pass Meetha Magic 11.75g Hanger      155
Rajnigandha 17g Zipper IC                 152
TZ 00 10g TIN                             136
Center Fruit                              108
Jabsons Hing Jeera Peanuts                101
Pulse Kachha Aam 175 Candy                 95
Bingo Tedhe Medhe                          91
Center Fresh                               79
Jabsons Green Peas onion & Garlic          65
Alpenliebe Creamfills Butter Toffee        64
Pulse Kachha Aam Jar (250 Candy)           61
Haldiram - Salted Peanuts (24 pcs)         58
Name: effective_qty, dtype: int64

In [9]:
matrix.shape

(1350, 159)

In [10]:
top_products.head(10)

itemName
Rajnigandha 17g Zipper 1 Pcs             1327
TZ 00 (with Silver) 0.45g                 615
Rajnigandha 4g                            486
TZ 00 4.25g (6 Pcs)                       277
RG Pearl Elaichi MP Pouch 0.14g           260
Rajnigandha 100g                          233
RG Pearl Elaichi 1.4g Hanger              188
Pass Pass Meetha Magic 2.2g (100 pcs)     169
Pass Pass Meetha Magic 11.75g Hanger      155
Rajnigandha 17g Zipper IC                 152
Name: effective_qty, dtype: int64